# Lab3.1 — `pynq.allocate()` și buffere pentru PS–PL

În aplicațiile accelerate, datele trebuie transferate eficient între memoria DDR utilizată de Processing System și blocurile hardware din Programmable Logic.

`pynq.allocate()` creează buffere NumPy alocate dintr-o zonă de memorie potrivită pentru accesul dispozitivelor hardware.


## Obiective

- crearea unui buffer cu `pynq.allocate()`;
- utilizarea bufferului ca array NumPy;
- diferențierea dintre adresa virtuală și adresa utilizată de hardware;
- observarea dimensiunii și tipului de date;
- pregătirea pentru transferurile DMA.


## 1. Array NumPy obișnuit

Începem cu un array NumPy obișnuit, utilizat de software-ul care rulează sub Linux.


In [ ]:
import numpy as np

a = np.arange(16, dtype=np.uint32)
print(a)
print("dtype  :", a.dtype)
print("nbytes :", a.nbytes)

## 2. Buffer PYNQ

Creăm acum un buffer cu aceeași formă și același tip de date.


In [ ]:
from pynq import allocate

buffer = allocate(shape=(16,), dtype=np.uint32)
buffer[:] = np.arange(16, dtype=np.uint32)

print(buffer)
print("dtype  :", buffer.dtype)
print("nbytes :", buffer.nbytes)

## 3. Adresele bufferului

Software-ul Linux lucrează cu adrese virtuale. Un accelerator hardware sau un DMA are nevoie de o adresă de memorie accesibilă dispozitivului.


In [ ]:
if hasattr(buffer, "device_address"):
    print("Device address :", hex(buffer.device_address))
elif hasattr(buffer, "physical_address"):
    print("Physical address:", hex(buffer.physical_address))

if hasattr(buffer, "virtual_address"):
    print("Virtual address :", hex(buffer.virtual_address))

## 4. Bufferul se utilizează ca un array NumPy


In [ ]:
buffer *= 2
print(buffer)

print("Sum  =", int(buffer.sum()))
print("Mean =", float(buffer.mean()))

## 5. Coerența datelor

Pe platformele unde memoria nu este coerentă automat cu dispozitivul hardware, bufferul PYNQ oferă operații de sincronizare a cache-ului:

- `flush()` — propagă modificările realizate de CPU către memorie înainte ca hardware-ul să citească datele;
- `invalidate()` — invalidează copia din cache înainte ca CPU să citească date modificate de hardware.

Aceste operații vor deveni relevante în transferurile DMA.


In [ ]:
print("flush available      :", hasattr(buffer, "flush"))
print("invalidate available :", hasattr(buffer, "invalidate"))

## 6. Buffere de dimensiuni diferite

Calculați memoria ocupată de buffere cu 256, 1024 și 65536 elemente `uint32`.


In [ ]:
for n in (256, 1024, 65536):
    b = allocate(shape=(n,), dtype=np.uint32)
    print(f"{n:7d} elements -> {b.nbytes:8d} bytes = {b.nbytes/1024:8.2f} KiB")
    del b

## 7. Eliberarea bufferului


In [ ]:
del buffer
print("Buffer released.")

## 8. Exercițiu

Creați un buffer cu **1024 elemente `float32`**, inițializați-l cu valori între 0 și 1 și afișați:

- dimensiunea în bytes;
- adresa pentru dispozitiv;
- primele 10 valori;
- valoarea medie.


In [ ]:
# Scrieți aici soluția


## 9. Întrebări

1. De ce un accelerator hardware nu poate utiliza pur și simplu o adresă virtuală Linux?
2. Ce avantaj oferă `allocate()` pentru comunicația PS–PL?
3. De ce este important tipul de date (`dtype`) al bufferului?
4. Care este rolul operațiilor `flush()` și `invalidate()`?
